<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week09-mimo.ipynb)

# Week 9 — Multiple-Input, Multiple-Output (MIMO)
**ECE 512 Wireless Communications** · companion notebook to the Week 9 lecture

This notebook follows the lecture's sections:

1. MIMO channel representation: $\mathbf{y} = \mathbf{H}\mathbf{x} + \mathbf{n}$, spatial multiplexing and invertibility
2. MIMO channel metrics: rank, SVD, Frobenius norm, condition number
3. MIMO channel models: Kronecker correlation and Ricean MIMO
4. Capacity: open loop, outage, and closed loop (water-filling)
5. MIMO receivers: zero-forcing, MMSE, maximum likelihood
6. Space-time coding vs. spatial multiplexing (Alamouti)

Throughout, $\mathbf{H}$ is the $M_R \times M_T$ channel matrix (rows = receive antennas),
$\mathbf{H}_w$ has i.i.d. $\mathcal{CN}(0,1)$ entries, $E_s/N_0$ is the **total** transmit SNR, and each transmit
antenna gets $E_s/M_T$ (open loop). Run the setup cell first. In Colab: *Runtime → Run all*.

In [ ]:
# --- Setup: installs the course package in Google Colab (safe to re-run) ---
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
import importlib.util
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider
from ece512 import mimo, modulation as mod
from ece512.fading import iid_rayleigh

db = lambda x: 10 * np.log10(x)

## 1. MIMO channel representation
For a frequency-flat, time-invariant channel, one channel use of an $M_R \times M_T$ link is

$$
\begin{bmatrix} y_1 \\ \vdots \\ y_{M_R} \end{bmatrix}
= \begin{bmatrix} h_{1,1} & \cdots & h_{1,M_T} \\ \vdots & & \vdots \\ h_{M_R,1} & \cdots & h_{M_R,M_T} \end{bmatrix}
\begin{bmatrix} x_1 \\ \vdots \\ x_{M_T} \end{bmatrix}
+ \begin{bmatrix} n_1 \\ \vdots \\ n_{M_R} \end{bmatrix}
\qquad\Longleftrightarrow\qquad \mathbf{y} = \mathbf{H}\mathbf{x} + \mathbf{n}.
$$

**Spatial multiplexing, $3\times 3$** (lecture exercises #1–#2): each antenna sends its own 4-QAM stream. Every receive
antenna hears a *mixture* of all three streams. If $\mathbf{H}$ is known (channel training) and invertible,
$\hat{\mathbf{x}} = \mathbf{H}^{-1}\mathbf{y}$ separates them again.

In [ ]:
qpsk = mod.constellation("psk", 4)                  # 4-QAM = QPSK, unit energy
H = iid_rayleigh((3, 3), rng=9)                      # one channel realization (known at RX)
idx, x = mod.random_symbols(qpsk, 3 * 300, rng=1)
X = x.reshape(3, -1)                                 # 3 streams x 300 channel uses
N0 = 10 ** (-25 / 10)                                # 25 dB SNR per stream
Y = H @ X + np.sqrt(N0 / 2) * (np.random.default_rng(2).standard_normal(X.shape)
                               + 1j * np.random.default_rng(3).standard_normal(X.shape))
X_hat = np.linalg.inv(H) @ Y

np.set_printoptions(precision=2, suppress=True)
print("H =\n", H)
print(f"det(H) = {np.linalg.det(H):.2f},  rank(H) = {mimo.rank(H)}")

colors = ["C0", "C1", "C2"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for k in range(3):
    axes[0].plot(Y[k].real, Y[k].imag, ".", ms=3, color=colors[k], label=f"$y_{k+1}$")
    axes[1].plot(X_hat[k].real, X_hat[k].imag, ".", ms=3, color=colors[k], label=f"$\\hat{{x}}_{k+1}$")
axes[1].plot(qpsk.real, qpsk.imag, "k+", ms=14, mew=2, label="4-QAM points")
axes[0].set_title("received $\\mathbf{y}$: every antenna sees a mixture")
axes[1].set_title("$\\hat{\\mathbf{x}} = \\mathbf{H}^{-1}\\mathbf{y}$: streams separated")
for ax in axes:
    ax.set_xlabel("in-phase"); ax.set_ylabel("quadrature"); ax.set_aspect("equal")
    ax.grid(True, alpha=0.3); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

**Invertibility and rank.** $\mathbf{H}$ must be non-singular: $\det(\mathbf{H}) = 0$ iff $\operatorname{rank}(\mathbf{H}) < M$.
Without multipath (e.g. a pure line-of-sight channel between closely spaced arrays) $\mathbf{H}$ is proportional to the
all-ones matrix — rank one. The slider $\alpha$ blends a rich-scattering channel into that rank-deficient one,
$\mathbf{H}(\alpha) = (1-\alpha)\,\mathbf{H}_w + \alpha\,\mathbf{1}\mathbf{1}^T$. Watch the smallest singular value
collapse and the zero-forcing output $\mathbf{H}^{\dagger}\mathbf{y}$ fill with amplified noise.

In [ ]:
H_w3 = iid_rayleigh((3, 3), rng=9)

def show_rank(alpha=0.5, snr_db=25.0):
    H = (1 - alpha) * H_w3 + alpha * np.ones((3, 3))
    rng = np.random.default_rng(4)
    X = qpsk[rng.integers(0, 4, (3, 300))]
    N0 = 10 ** (-snr_db / 10)
    Y = H @ X + np.sqrt(N0 / 2) * (rng.standard_normal(X.shape) + 1j * rng.standard_normal(X.shape))
    X_hat = np.linalg.pinv(H) @ Y
    s = mimo.singular_values(H)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [1, 1.2]})
    axes[0].bar([1, 2, 3], s, color="C0")
    axes[0].set_yscale("log"); axes[0].set_ylim(1e-3, 10)
    axes[0].set_xticks([1, 2, 3]); axes[0].set_xticklabels(["$\\sigma_1$", "$\\sigma_2$", "$\\sigma_3$"])
    axes[0].set_ylabel("singular value of $\\mathbf{H}$")
    kappa = "$\\infty$" if s[-1] < 1e-12 * s[0] else f"{s[0] / s[-1]:.3g}"
    axes[0].set_title(f"$|\\det\\mathbf{{H}}|$ = {abs(np.linalg.det(H)):.3f},  "
                      f"rank = {mimo.rank(H, 1e-6)},  $\\kappa$ = {kappa}")
    axes[0].grid(True, axis="y", alpha=0.3)
    for k in range(3):
        axes[1].plot(X_hat[k].real, X_hat[k].imag, ".", ms=3, color=colors[k], label=f"$\\hat{{x}}_{k+1}$")
    axes[1].plot(qpsk.real, qpsk.imag, "k+", ms=14, mew=2)
    axes[1].set_xlim(-3, 3); axes[1].set_ylim(-3, 3); axes[1].set_aspect("equal")
    axes[1].set_xlabel("in-phase"); axes[1].set_ylabel("quadrature")
    axes[1].set_title(f"zero-forcing output, $\\alpha$ = {alpha:.2f}, SNR = {snr_db:.0f} dB")
    axes[1].grid(True, alpha=0.3); axes[1].legend(loc="upper right", fontsize=8)
    plt.tight_layout(); plt.show()

interact(show_rank, alpha=FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05),
         snr_db=FloatSlider(value=25.0, min=0.0, max=40.0, step=5.0));

**Frequency-selective MIMO.** An $L$-tap channel $\mathbf{y}(k) = \sum_{l=0}^{L-1}\mathbf{H}_l\,\mathbf{x}(k-l)$ becomes,
with OFDM, $N$ narrowband channels $\mathbf{y}_i = \mathbf{H}_i\mathbf{x}_i + \mathbf{n}_i$ with
$\mathbf{H}_i = \sum_l \mathbf{H}_l e^{-j2\pi l i/N}$. Each subcarrier is a flat MIMO channel with its own singular
values — and its own mutual information (Section 4).

In [ ]:
L, N_sc = 6, 64
pdp = np.exp(-np.arange(L) / 2.0); pdp /= pdp.sum()          # exponential power-delay profile
H_taps = iid_rayleigh((L, 2, 2), rng=5) * np.sqrt(pdp)[:, None, None]
H_sc = mimo.frequency_response(H_taps, N_sc)                  # (N, M_R, M_T)
s_sc = mimo.singular_values(H_sc)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(db(s_sc ** 2), marker=".")
axes[0].set_ylabel("$\\lambda_i$ of $\\mathbf{H}_i\\mathbf{H}_i^H$ (dB)")
axes[0].legend(["$\\lambda_1$ (strong mode)", "$\\lambda_2$ (weak mode)"])
axes[1].plot(mimo.capacity_open_loop(H_sc, 20), marker=".", color="C2")
axes[1].set_ylabel("mutual information (bit/s/Hz)")
axes[1].set_title("$2\\times 2$, $E_s/N_0$ = 20 dB")
axes[0].set_title(f"{L}-tap $2\\times2$ channel, N = {N_sc} subcarriers")
for ax in axes:
    ax.set_xlabel("subcarrier index i"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. MIMO channel metrics
**SVD.** $\mathbf{H} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^H$ with $\boldsymbol{\Sigma} = \operatorname{diag}\{\sigma_i\}_{i=1}^r$,
and $\mathbf{H}\mathbf{H}^H = \mathbf{Q}\boldsymbol{\Lambda}\mathbf{Q}^H$ with eigenvalues $\lambda_i = \sigma_i^2$.
The rank $r$ is the number of non-zero $\sigma_i$.

| metric | definition | tells you about |
|---|---|---|
| Frobenius norm | $\lVert\mathbf{H}\rVert_F^2 = \operatorname{Tr}(\mathbf{H}\mathbf{H}^H) = \sum_{i,j}\lvert h_{i,j}\rvert^2 = \sum_i \lambda_i$ | total power gain (STBC performance) |
| largest $\sigma_{\max}$ | | optimum beamforming gain $\lambda_{\max}$ |
| smallest $\sigma_{\min}$ | | zero-forcing performance in spatial multiplexing |
| condition number | $\kappa = \sigma_{\max}/\sigma_{\min} = \sqrt{\lambda_{\max}/\lambda_{\min}}$ | small → multiplex, large → beamform |
| Demmel condition number | $\kappa_D = \lVert\mathbf{H}\rVert_F/\sigma_{\min}$ | same, normalised by total power |

*Notation note:* as on the slides, $\lambda_i$ are the eigenvalues of $\mathbf{H}\mathbf{H}^H$ and $\sigma_i=\sqrt{\lambda_i}$ the singular values of $\mathbf{H}$; the condition numbers are $\kappa=\sigma_{\max}/\sigma_{\min}$ and $\kappa_D=\Vert\mathbf{H}\Vert_F/\sigma_{\min}$.

In [ ]:
H = iid_rayleigh((4, 4), rng=11)
lam = mimo.eigenvalues(H)
print("lambda_i (eigenvalues of H H^H):", np.round(lam, 3))
print(f"||H||_F^2 = {mimo.frobenius_sq(H):.4f}   Tr(H H^H) = {np.trace(H @ H.conj().T).real:.4f}"
      f"   sum lambda_i = {lam.sum():.4f}")
print(f"kappa = {mimo.condition_number(H):.2f}   kappa_D = {mimo.demmel_condition_number(H):.2f}"
      f"   rank = {mimo.rank(H)}")

Over an ensemble of i.i.d. Rayleigh channels the metrics are random variables. For $\mathbf{H}_w$ the squared
Frobenius norm is a sum of $M_T M_R$ unit-mean exponentials, i.e. Gamma$(M_TM_R, 1)$ — the same statistic as
$M_TM_R$-branch MRC. The eigenvalues spread out over several orders of magnitude; correlation and a strong LOS
component make the spread (and $\kappa$) much larger.

In [ ]:
from scipy.stats import gamma as gamma_dist
n_mc, M = 20000, 4
Hw = iid_rayleigh((n_mc, M, M), rng=12)
lam = mimo.eigenvalues(Hw)
R09 = mimo.exponential_correlation(M, 0.9)
ensembles = {
    "i.i.d. Rayleigh": Hw,
    "Kronecker, $\\rho$ = 0.5": mimo.kronecker_channel(*(2 * [mimo.exponential_correlation(M, 0.5)]), H_w=Hw),
    "Kronecker, $\\rho$ = 0.9": mimo.kronecker_channel(R09, R09, H_w=Hw),
    "Ricean, K = 10 dB": mimo.ricean_channel(10.0, mimo.los_channel(M, M), Hw),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
bins = np.linspace(-40, 20, 121)
for i in range(M):
    axes[0].hist(db(lam[:, i]), bins=bins, density=True, alpha=0.6, label=f"$\\lambda_{i+1}$")
axes[0].set_xlabel("ordered eigenvalue of $\\mathbf{H}\\mathbf{H}^H$ (dB)"); axes[0].set_ylabel("pdf (1/dB)")
axes[0].set_title("4×4 i.i.d. Rayleigh: ordered eigenvalues"); axes[0].legend()

f2 = mimo.frobenius_sq(Hw)
axes[1].hist(f2, bins=60, density=True, alpha=0.6, label="simulation")
xx = np.linspace(0, f2.max(), 300)
axes[1].plot(xx, gamma_dist.pdf(xx, M * M), "k", label="Gamma($M_TM_R$, 1) theory")
axes[1].set_xlabel("$\\|\\mathbf{H}\\|_F^2$"); axes[1].set_ylabel("pdf")
axes[1].set_title(f"Frobenius norm, mean = {f2.mean():.2f} (theory {M * M})"); axes[1].legend()

for name, Hs in ensembles.items():
    k = np.sort(mimo.condition_number(Hs))
    axes[2].semilogx(k, np.arange(1, len(k) + 1) / len(k), label=name)
axes[2].set_xlabel("condition number $\\kappa = \\sigma_{\\max}/\\sigma_{\\min}$")
axes[2].set_ylabel("$\\Pr(\\kappa \\leq$ abscissa$)$"); axes[2].set_title("condition number CDF, 4×4")
axes[2].set_xlim(1, 1e4); axes[2].legend(fontsize=8)
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. MIMO channel models
**Kronecker model.** Transmit and receive correlation are modelled separately,

$$\mathbf{H}_{\mathrm{kron}} = \mathbf{R}_r^{1/2}\,\mathbf{H}_w\,\mathbf{R}_t^{1/2}, \qquad
\mathbf{R} = \mathbf{R}_t^T \otimes \mathbf{R}_r .$$

A common choice for uniform linear arrays is the *exponential correlation* model $[\mathbf{R}]_{i,j} = \rho^{|i-j|}$
(here $\rho$ is a correlation coefficient, not the SNR). Closely spaced elements → $\rho \to 1$ → the channel loses
rank. Move the sliders: the weak eigenmodes sink, and the open-loop capacity distribution shifts left.

In [ ]:
n_k, M, snr_k = 4000, 4, 15
Hw_k = iid_rayleigh((n_k, M, M), rng=13)
C_iid = np.sort(mimo.capacity_open_loop(Hw_k, snr_k))
lam_iid = mimo.eigenvalues(Hw_k).mean(axis=0)

def show_kronecker(rho_r=0.7, rho_t=0.7):
    R_r, R_t = mimo.exponential_correlation(M, rho_r), mimo.exponential_correlation(M, rho_t)
    Hk = mimo.kronecker_channel(R_r, R_t, H_w=Hw_k)
    lam_k = mimo.eigenvalues(Hk).mean(axis=0)
    C_k = np.sort(mimo.capacity_open_loop(Hk, snr_k))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [0.8, 1, 1.1]})
    im = axes[0].imshow(np.abs(np.kron(R_t.T, R_r)), vmin=0, vmax=1, cmap="viridis")
    axes[0].set_title("$|\\mathbf{R}| = |\\mathbf{R}_t^T \\otimes \\mathbf{R}_r|$ (16×16)")
    axes[0].set_xticks([]); axes[0].set_yticks([]); fig.colorbar(im, ax=axes[0], shrink=0.8)
    i = np.arange(1, M + 1)
    axes[1].bar(i - 0.2, db(lam_iid) + 30, 0.4, label="i.i.d.", bottom=-30)
    axes[1].bar(i + 0.2, db(lam_k) + 30, 0.4, label="Kronecker", bottom=-30)
    axes[1].set_ylim(-30, 15); axes[1].set_xticks(i)
    axes[1].set_xticklabels([f"$\\lambda_{k}$" for k in i])
    axes[1].set_ylabel("mean eigenvalue $\\mathrm{E}[\\lambda_i]$ (dB)")
    axes[1].set_title(f"eigenvalue spread: $\\lambda_1/\\lambda_4$ = {db(lam_k[0] / lam_k[-1]):.1f} dB "
                      f"(i.i.d. {db(lam_iid[0] / lam_iid[-1]):.1f} dB)", fontsize=10)
    axes[1].legend()
    p = np.arange(1, n_k + 1) / n_k
    axes[2].plot(C_iid, p, label=f"i.i.d.: mean {C_iid.mean():.2f}")
    axes[2].plot(C_k, p, label=f"$\\rho_r$={rho_r:.2f}, $\\rho_t$={rho_t:.2f}: mean {C_k.mean():.2f}")
    axes[2].set_xlabel("open-loop mutual information $I$ (bit/s/Hz)"); axes[2].set_ylabel("CDF")
    axes[2].set_title(f"4×4, $E_s/N_0$ = {snr_k} dB"); axes[2].set_xlim(0, 24)
    axes[2].legend(loc="lower right", fontsize=9)
    for ax in axes[1:]:
        ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_kronecker, rho_r=FloatSlider(value=0.7, min=0.0, max=0.99, step=0.05),
         rho_t=FloatSlider(value=0.7, min=0.0, max=0.99, step=0.05));

**Estimating the model from "measurements"** (lecture: *Model Parameter Estimation*). From $N$ snapshots,
$\hat{\mathbf{R}}_{\mathrm{RX}} = \frac{1}{N}\sum_i \mathbf{H}(i)\mathbf{H}(i)^H$ and
$\hat{\mathbf{R}}_{\mathrm{TX}} = \frac{1}{N}\sum_i \mathbf{H}(i)^H\mathbf{H}(i)$. For the Kronecker model
$\mathrm{E}[\mathbf{H}\mathbf{H}^H] = \operatorname{Tr}(\mathbf{R}_t)\,\mathbf{R}_r = M_T\mathbf{R}_r$, so dividing by $M_T$ recovers $\mathbf{R}_r$.

In [ ]:
R_r_true = mimo.exponential_correlation(M, 0.8)
H_meas = mimo.kronecker_channel(R_r_true, np.eye(M), n=5000, rng=14)
R_rx, R_tx = mimo.estimate_correlations(H_meas)
print("true R_r:\n", R_r_true)
print("estimate R_RX / M_T from 5000 snapshots:\n", (R_rx / M).real)
print("estimate R_TX / M_R (should be ~ I):\n", (R_tx / M).real)

**Ricean MIMO.** A line-of-sight component superposes a *deterministic* matrix on the scattered part,

$$\mathbf{H}_{\mathrm{rice}} = \sqrt{\tfrac{1}{1+K}}\,\mathbf{H}_{\mathrm{rayleigh}} + \sqrt{\tfrac{K}{1+K}}\,\mathbf{H}_{\mathrm{LOS}},$$

with $\mathbf{H}_{\mathrm{LOS}} = \mathbf{a}_R(\theta_R)\,\mathbf{a}_T(\theta_T)^T$ (far-field, rank one; all-ones at broadside).
The average power $\mathrm{E}\lVert\mathbf{H}\rVert_F^2 = M_TM_R$ does not depend on $K$, yet as $K$ grows the power
concentrates in *one* eigenmode: the eigenvalue spread grows and capacity drops, even though the channel "fades less".

In [ ]:
K_db = np.arange(-10, 26, 2.5)
Hw_r = iid_rayleigh((3000, M, M), rng=15)
H_los = mimo.los_channel(M, M)
lam_K, C_K = [], {10: [], 25: []}
for kdb in K_db:
    Hr = mimo.ricean_channel(10 ** (kdb / 10), H_los, Hw_r)
    lam_K.append(mimo.eigenvalues(Hr).mean(axis=0))
    for s in C_K:
        C_K[s].append(mimo.capacity_open_loop(Hr, s).mean())
lam_K = np.array(lam_K)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i in range(M):
    axes[0].plot(K_db, db(lam_K[:, i]), marker=".", label=f"$\\mathrm{{E}}[\\lambda_{i+1}]$")
axes[0].axhline(db(M * M), color="gray", ls=":", label="$M_TM_R$ = 16 (all power in one mode)")
axes[0].set_ylabel("mean ordered eigenvalue (dB)"); axes[0].set_title("4×4 Ricean MIMO: eigenvalues vs K")
for s, c in C_K.items():
    axes[1].plot(K_db, c, marker=".", label=f"$E_s/N_0$ = {s} dB")
    axes[1].axhline(np.log2(1 + 10 ** (s / 10) / M * M * M), ls=":", color="gray")
axes[1].plot([], [], ":", color="gray", label="pure LOS limit $\\log_2(1 + M_R E_s/N_0)$")
axes[1].set_ylabel("ergodic capacity (bit/s/Hz)"); axes[1].set_title("open-loop capacity vs K")
for ax in axes:
    ax.set_xlabel("Ricean K-factor (dB)"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Capacity
### 4a. Open-loop (ergodic) capacity
With no channel knowledge at the transmitter, power is split equally over the $M_T$ antennas:

$$I = \log_2\det\!\left(\mathbf{I}_{M_R} + \frac{E_s}{M_T N_0}\,\mathbf{H}\mathbf{H}^H\right)
    = \sum_{i=1}^{r}\log_2\!\left(1 + \frac{E_s}{M_T N_0}\,\lambda_i\right).$$

The ergodic capacity is $C = \mathrm{E}[I]$. Solid lines are the exact i.i.d. Rayleigh result (Telatar's Laguerre
integral, `mimo.ergodic_capacity_iid`); markers are Monte Carlo averages. At high SNR each of the
$\min(M_T, M_R)$ spatial "pipes" adds $\log_2 10 \approx 3.32$ bit/s/Hz per 10 dB.

In [ ]:
snr = np.arange(-10, 41, 5)
configs = [(1, 1), (4, 1), (1, 4), (2, 2), (4, 4)]     # (M_T, M_R)
fig, ax = plt.subplots(figsize=(8.5, 5))
print(f"{'M_T x M_R':>10} {'slope 30->40 dB (bit/s/Hz per 10 dB)':>38} {'min(M_T,M_R)*log2(10)':>22}")
for k, (mt, mr) in enumerate(configs):
    H = iid_rayleigh((3000, mr, mt), rng=20 + k)
    mc = mimo.capacity_open_loop(H, snr).mean(axis=1)
    th = mimo.ergodic_capacity_iid(mt, mr, snr)
    ax.plot(snr, th, color=f"C{k}", label=f"$M_T$={mt}, $M_R$={mr}")
    ax.plot(snr, mc, "o", color=f"C{k}", ms=4)
    print(f"{mt:>4} x {mr:<4} {th[-1] - th[-3]:>38.2f} {min(mt, mr) * np.log2(10):>22.2f}")
ax.plot([], [], "ko", ms=4, label="Monte Carlo")
ax.set_xlabel("$E_s/N_0$ (dB)"); ax.set_ylabel("ergodic capacity (bit/s/Hz)")
ax.set_title("open-loop ergodic capacity, i.i.d. Rayleigh")
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

Note the $M_T=4, M_R=1$ vs. $M_T=1, M_R=4$ pair: both have one spatial pipe (same slope), but the receive array
collects $M_R$ times more energy (array gain, a 6 dB shift) while the uninformed transmit array only averages out
the fading (diversity), splitting $E_s$ over antennas whose contributions add incoherently.

### 4b. Outage capacity
Because $\mathbf{H}$ is random, $I$ is a random variable. The outage capacity at level $p$ is the rate supported by
$(100-p)\%$ of channel realizations:

$$\Pr\!\left(I \le C_{\mathrm{out},p}\right) = p\%.$$

In [ ]:
H_out = {cfg: iid_rayleigh((5000, cfg[1], cfg[0]), rng=30 + k) for k, cfg in enumerate(configs)}

def show_outage(snr_db=10.0, p_percent=10.0):
    fig, ax = plt.subplots(figsize=(9, 4.8))
    p = p_percent / 100
    for k, (cfg, H) in enumerate(H_out.items()):
        I = np.sort(mimo.capacity_open_loop(H, snr_db))
        c_out = mimo.outage_capacity(I, p)
        ax.plot(I, np.arange(1, len(I) + 1) / len(I), color=f"C{k}",
                label=f"$M_T$={cfg[0]}, $M_R$={cfg[1]}: $C_{{\\mathrm{{out}},{p_percent:g}}}$ = {c_out:.2f}, "
                      f"$\\mathrm{{E}}[I]$ = {I.mean():.2f}")
        ax.plot(c_out, p, "o", color=f"C{k}")
    ax.axhline(p, color="k", ls="--", lw=1)
    ax.set_yscale("log"); ax.set_ylim(1e-3, 1)
    ax.set_xlabel("mutual information $I$ (bit/s/Hz)"); ax.set_ylabel("$\\Pr(I \\leq$ abscissa$)$")
    ax.set_title(f"CDF of open-loop mutual information, $E_s/N_0$ = {snr_db:.0f} dB")
    ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout(); plt.show()

interact(show_outage, snr_db=FloatSlider(value=10.0, min=-5.0, max=30.0, step=2.5),
         p_percent=FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0));

### 4c. Closed-loop capacity: water-filling
With $\mathbf{H} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^H$ known at the transmitter, precoding $\mathbf{x} = \mathbf{V}\tilde{\mathbf{x}}$
and receive shaping $\tilde{\mathbf{y}} = \mathbf{U}^H\mathbf{y}$ give $r$ parallel modes
$\tilde y_i = \sqrt{E_s/M_T}\,\sqrt{\lambda_i}\,\tilde x_i + n_i$. Allocating power $\gamma_i$ to mode $i$,

$$C = \max_{\sum_i \gamma_i = M_T}\ \sum_{i=1}^{r}\log_2\!\left(1 + \frac{E_s\gamma_i}{M_T N_0}\lambda_i\right),
\qquad \gamma_i^{\mathrm{opt}} = \left(\mu - \frac{M_T N_0}{E_s\lambda_i}\right)^{+}.$$

Picture the "vessel floor" of mode $i$ at height $M_TN_0/(E_s\lambda_i)$ and pour $M_T$ units of water: the level $\mu$
settles where the total is $M_T$. Weak modes (high floor) get nothing at low SNR. The right panel shows the per-mode
power for the same channel over all SNRs.

In [ ]:
M_wf = 4
snr_sweep = np.arange(-15, 35.1, 0.5)

def show_waterfill(snr_db=5.0, rho=0.5, seed=1):
    R = mimo.exponential_correlation(M_wf, rho)
    H = mimo.kronecker_channel(R, R, rng=seed)
    lam = mimo.eigenvalues(H)
    g, mu = mimo.waterfill(lam, snr_db, M_wf)
    floor = M_wf / (10 ** (snr_db / 10) * lam)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
    i = np.arange(1, M_wf + 1)
    top = max(mu, np.sort(floor)[0]) * 1.6
    axes[0].bar(i, np.minimum(floor, 10 * top), color="0.55", label="floor $M_TN_0/(E_s\\lambda_i)$")
    axes[0].bar(i, g, bottom=floor, color="C0", alpha=0.8, label="power $\\gamma_i$")
    axes[0].axhline(mu, color="C0", ls="--", label=f"water level $\\mu$ = {mu:.2f}")
    axes[0].set_ylim(0, top); axes[0].set_xticks(i)
    axes[0].set_xticklabels([f"mode {k}\n$\\lambda_{k}$={l:.2g}\n$\\gamma_{k}$={gk:.2f}"
                             for k, l, gk in zip(i, lam, g)])
    C_cl = np.sum(np.log2(1 + 10 ** (snr_db / 10) * g * lam / M_wf))
    C_ol = mimo.capacity_open_loop(H, snr_db)
    axes[0].set_title(f"$E_s/N_0$ = {snr_db:.1f} dB: closed loop {C_cl:.2f} vs open loop {C_ol:.2f} bit/s/Hz",
                      fontsize=10)
    axes[0].legend(fontsize=8, loc="upper left")
    G = np.array([mimo.waterfill(lam, s, M_wf)[0] for s in snr_sweep])
    axes[1].stackplot(snr_sweep, G.T, labels=[f"mode {k}" for k in i], alpha=0.8)
    axes[1].axvline(snr_db, color="k", ls="--")
    axes[1].set_xlabel("$E_s/N_0$ (dB)"); axes[1].set_ylabel("power $\\gamma_i$ (sum = $M_T$)")
    axes[1].set_ylim(0, M_wf); axes[1].set_xlim(snr_sweep[0], snr_sweep[-1])
    axes[1].set_title("water-filling allocation vs SNR ($\\gamma_i \\to 1$ as SNR grows)")
    axes[1].legend(loc="upper right", fontsize=8); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_waterfill, snr_db=FloatSlider(value=5.0, min=-15.0, max=30.0, step=1.0),
         rho=FloatSlider(value=0.5, min=0.0, max=0.95, step=0.05), seed=IntSlider(value=1, min=0, max=20));

### 4d. Open vs. closed loop
Channel knowledge at the transmitter helps most at **low SNR** (pour everything into the best mode) and when the
channel is **rank-deficient or correlated** (don't waste power on modes that cannot carry data), or when $M_T > M_R$
(an uninformed transmitter spreads power over $M_T$ directions but only $M_R$ exist). At high SNR in a
well-conditioned square channel both converge.

In [ ]:
snr_c = np.arange(-10, 31, 5)
cases = {
    "4×4 i.i.d.": iid_rayleigh((1000, 4, 4), rng=40),
    "4×4 Kronecker $\\rho$=0.9": mimo.kronecker_channel(R09, R09, n=1000, rng=41),
    "$M_T$=4, $M_R$=2 i.i.d.": iid_rayleigh((1000, 2, 4), rng=42),
}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for k, (name, H) in enumerate(cases.items()):
    ol = mimo.capacity_open_loop(H, snr_c).mean(axis=1)
    cl = np.array([mimo.capacity_closed_loop(H, s).mean() for s in snr_c])
    axes[0].plot(snr_c, cl, "-o", color=f"C{k}", ms=4, label=f"{name}, closed loop")
    axes[0].plot(snr_c, ol, "--", color=f"C{k}", label=f"{name}, open loop")
    axes[1].plot(snr_c, cl - ol, "-o", color=f"C{k}", ms=4, label=name)
axes[0].set_ylabel("ergodic capacity (bit/s/Hz)"); axes[0].set_title("closed (water-filling) vs open loop")
axes[1].set_ylabel("closed − open loop (bit/s/Hz)"); axes[1].set_title("benefit of transmit channel knowledge")
for ax in axes:
    ax.set_xlabel("$E_s/N_0$ (dB)"); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. MIMO receivers for spatial multiplexing
$2\times2$ spatial multiplexing, QPSK on each antenna (4 bit/s/Hz), signal model
$\mathbf{y} = \sqrt{E_s/M_T}\,\mathbf{H}\mathbf{x} + \mathbf{n}$ (write $\mathbf{G} = \sqrt{E_s/M_T}\,\mathbf{H}$):

* **Zero-forcing:** $\hat{\mathbf{x}} = \mathbf{G}^{\dagger}\mathbf{y}$, $\mathbf{G}^{\dagger} = (\mathbf{G}^H\mathbf{G})^{-1}\mathbf{G}^H$ — removes interference, enhances noise.
* **MMSE:** $\hat{\mathbf{x}} = \mathbf{G}^H(\mathbf{G}\mathbf{G}^H + \mathbf{R}_n)^{-1}\mathbf{y}$ with $\mathbf{R}_n = N_0\mathbf{I}$ — trades residual interference for less noise.
* **Maximum likelihood:** $\hat{\mathbf{x}} = \arg\min_{\mathbf{x}}\lVert\mathbf{y} - \mathbf{G}\mathbf{x}\rVert$ over all $4^{M_T} = 16$ candidate vectors.

**Theory check (ZF).** In i.i.d. Rayleigh fading each ZF stream's post-processing SNR is
$\frac{E_s}{M_TN_0}\big/[(\mathbf{H}^H\mathbf{H})^{-1}]_{kk}$, and $1/[(\mathbf{H}^H\mathbf{H})^{-1}]_{kk}$ is
Gamma$(M_R - M_T + 1)$: ZF behaves like $(M_R-M_T+1)$-branch MRC — for $2\times2$, **diversity one**, i.e. like a single
Rayleigh link. ML achieves diversity $M_R = 2$ and approaches the interference-free bound (each stream alone with
$M_R$-branch MRC).

In [ ]:
snr_b = np.arange(0, 25, 3)
n_b = 150000
rng = np.random.default_rng(50)
ber = {"ZF": [], "MMSE": [], "ML": []}
for s in snr_b:
    H = iid_rayleigh((n_b, 2, 2), rng)
    idx = rng.integers(0, 4, (n_b, 2))
    y = mimo.transmit_sm(H, qpsk[idx], s, rng)
    det = {"ZF": mod.detect(mimo.zf_equalize(H, y, s), qpsk),
           "MMSE": mod.detect(mimo.mmse_equalize(H, y, s), qpsk),
           "ML": mimo.ml_detect(H, y, s, qpsk)}
    for k in ber:
        ber[k].append(mod.bit_error_rate(idx.ravel(), det[k].ravel(), 4))

per_bit_db = snr_b - db(2 * 2)            # E_s/(M_T N_0) per stream, /2 bits per QPSK symbol
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for k, m in zip(ber, ["o", "s", "^"]):
    ax.semilogy(snr_b, np.maximum(ber[k], 1e-7), m, ms=6, label=f"{k} (Monte Carlo)")
ax.semilogy(snr_b, mimo.ber_bpsk_mrc(per_bit_db, 1), "C0-", label="ZF theory: Rayleigh, diversity $M_R-M_T+1$ = 1")
ax.semilogy(snr_b, mimo.ber_bpsk_mrc(per_bit_db, 2), "k--",
            label="interference-free bound: MRC, diversity $M_R$ = 2")
ax.set_ylim(1e-5, 0.5); ax.set_xlabel("$E_s/N_0$ (dB), total transmit SNR"); ax.set_ylabel("bit error rate")
ax.set_title("2×2 spatial multiplexing, QPSK per antenna (4 bit/s/Hz)")
ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 6. Space-time coding vs. spatial multiplexing
**Alamouti ($M_T = 2$).** Two symbols are sent over two slots:

$$\text{slot 1: } [s_1,\ s_2], \qquad \text{slot 2: } [-s_2^*,\ s_1^*], \qquad \text{each scaled by } \sqrt{E_s/2}.$$

Linear combining yields $\hat s_k = \sqrt{E_s/2}\,\lVert\mathbf{H}\rVert_F^2 s_k + \text{noise}$ — decoupled symbols,
full diversity $2M_R$ (the Frobenius norm is the relevant metric, Section 2), but only **one symbol per slot**.
Because the transmitter does not know $\mathbf{H}$ it must split $E_s$ over two antennas: Alamouti $2\times1$ has the
same diversity as $1\times2$ MRC but is 3 dB worse (no array gain).

In [ ]:
def ber_alamouti(m_r, points, s_db, n, rng):
    H = iid_rayleigh((n, m_r, 2), rng)
    idx = rng.integers(0, len(points), (n, 2))
    s_hat = mimo.alamouti_combine(H, mimo.alamouti_transmit(H, points[idx], s_db, rng), s_db)
    return mod.bit_error_rate(idx.ravel(), mod.detect(s_hat, points).ravel(), len(points))

snr_a = np.arange(0, 25, 3)
rng = np.random.default_rng(60)
sim_a21 = [ber_alamouti(1, qpsk, s, 200000, rng) for s in snr_a]
sim_a22 = [ber_alamouti(2, qpsk, s, 200000, rng) for s in snr_a]
pb = snr_a - db(2)                         # per-bit SNR of QPSK with all of E_s on one path
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.semilogy(snr_a, mimo.ber_bpsk_mrc(pb, 1), "k-", label="1×1 (no diversity)")
ax.semilogy(snr_a, mimo.ber_bpsk_mrc(pb, 2), "C1-", label="1×2 MRC: diversity 2, +3 dB array gain")
ax.semilogy(snr_a, mimo.ber_bpsk_mrc(pb - db(2), 2), "C0-", label="Alamouti 2×1 theory: diversity 2")
ax.semilogy(snr_a, mimo.ber_bpsk_mrc(pb - db(2), 4), "C2-", label="Alamouti 2×2 theory: diversity 4")
ax.semilogy(snr_a, sim_a21, "C0o", label="Alamouti 2×1 (Monte Carlo)")
ax.semilogy(snr_a, np.maximum(sim_a22, 1e-7), "C2s", label="Alamouti 2×2 (Monte Carlo)")
ax.set_ylim(1e-5, 0.5); ax.set_xlabel("$E_s/N_0$ (dB), total transmit SNR"); ax.set_ylabel("bit error rate")
ax.set_title("QPSK, 2 bit/s/Hz: transmit diversity with the Alamouti code")
ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**The trade-off at equal rate.** On a $2\times2$ link, compare schemes that all deliver **4 bit/s/Hz**:
Alamouti with 16-QAM (one symbol/slot, diversity 4) vs. spatial multiplexing with QPSK on each antenna
(two symbols/slot; diversity 2 with ML, 1 with ZF). Denser constellations cost SNR at first, but the higher diversity
order (steeper slope) wins at high SNR — the "sweet spot" between diversity and multiplexing depends on the SNR
regime and the receiver.

In [ ]:
qam16 = mod.constellation("qam", 16)
snr_t = np.arange(0, 31, 3)
rng = np.random.default_rng(70)
res = {"Alamouti, 16-QAM": [], "SM ML, QPSK": [], "SM ZF, QPSK": []}
n_t = 100000
for s in snr_t:
    res["Alamouti, 16-QAM"].append(ber_alamouti(2, qam16, s, n_t, rng))
    H = iid_rayleigh((n_t, 2, 2), rng)
    idx = rng.integers(0, 4, (n_t, 2))
    y = mimo.transmit_sm(H, qpsk[idx], s, rng)
    res["SM ML, QPSK"].append(mod.bit_error_rate(idx.ravel(), mimo.ml_detect(H, y, s, qpsk).ravel(), 4))
    res["SM ZF, QPSK"].append(mod.bit_error_rate(idx.ravel(),
                                                 mod.detect(mimo.zf_equalize(H, y, s), qpsk).ravel(), 4))
fig, ax = plt.subplots(figsize=(8.5, 5))
for (k, v), m in zip(res.items(), ["o-", "s-", "^-"]):
    ax.semilogy(snr_t, np.where(np.array(v) > 0, v, np.nan), m, ms=5, label=k)
ax.set_ylim(1e-5, 0.5); ax.set_xlabel("$E_s/N_0$ (dB), total transmit SNR"); ax.set_ylabel("bit error rate")
ax.set_title("2×2 at 4 bit/s/Hz: diversity (Alamouti) vs multiplexing (SM)")
ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## Try it yourself
* In Section 1, set $\alpha$ just below 1. Which singular value vanishes, and why do *all three* ZF outputs degrade
  rather than just one? How could the transmitter adapt to a rank-one channel (lecture exercise #2)?
* Section 3: with $\rho_r = 0$, how large must $\rho_t$ be before the 4×4 capacity at 15 dB falls by 1 bit/s/Hz?
  Is transmit or receive correlation more harmful when $M_T = M_R$? Try `mimo.keyhole_channel`: it is rank one
  even though every entry is Rayleigh — what does its capacity CDF look like?
* Section 4c: at what SNR does water-filling first turn on the 4th mode for $\rho = 0.9$? Relate that SNR to
  $\lambda_3$ and $\lambda_4$ using $\gamma_i^{\mathrm{opt}} = (\mu - M_TN_0/(E_s\lambda_i))^+$.
* Section 6: add an MMSE curve to the equal-rate comparison, and repeat it at 8 bit/s/Hz (Alamouti 256-QAM vs. SM
  16-QAM). Where do the curves cross now?